In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina' # high res plotting

import sys
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')


from spikeparam.patch.fit import Spike
from spikeparam.patch.fit import SpikeGroup



from neurodsp import spectral

from scipy import signal
import scipy

import h5py
from tqdm.notebook import tqdm

import numpy as np
import pandas as pd
import scipy as sp

from neurodsp import filt
from neurodsp.timefrequency import amp_by_time, phase_by_time
from neurodsp.plts import plot_time_series, plot_instantaneous_measure
from neurodsp.plts.time_series import plot_bursts
from neurodsp.burst import detect_bursts_dual_threshold, compute_burst_stats

from scipy.signal import sosfiltfilt, butter

from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats


import os

from fooof import FOOOF

sns.set(rc={'figure.figsize':(12,9)})
sns.set_style('whitegrid')
sns.set_style("whitegrid", {'axes.grid' : False})


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


/var/folders/rr/tk8wmt7x4xlgzgmcy768y1rm0000gn/T/ipykernel_8202/786254587.py:45: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOF


In [3]:
def butter_bandpass(data, fs, filt_freq, order):
    # make sure that the user provides two frequencies for bandpass filter
    try:
        # renormalize frequencies in Hz to fractional scale required by butter
        nyq = 0.5 * fs # nyquist frequency
        f0 = (filt_freq[0]/nyq)
        f1 = (filt_freq[1]/nyq)

        # highpass first
        sos = butter(order, f0, btype = 'high', analog=False, output='sos')
        y = sosfiltfilt(sos, data)

        # then lowpass
        sos = butter(order, f1, btype = 'low', analog=False, output='sos')
        y = sosfiltfilt(sos, y)
        
        return y
    
    except:
        print("filt_freq must have two frequencies for a bandpass filter")

In [4]:
### Figure out loading for patch files
### Add saving of time arrays 

def load_spe1_data(data_path, lpf_fs, npx_fs, all_patch_fs, cell_ids, npx_channels, npx_patch_channel, out_lfp_path, out_npx_path, out_patch_path):
    
    lfp_one_ms = float(lfp_fs / 1000)
    
    npx_one_ms = float(npx_fs/1000)


    lfp_recording = []
    lfp_times = []
    patch_filt = []
    patch_times = []
    
    #loop through lfp recording folder
    for filename in tqdm(os.listdir(data_path)):

        recfile = os.path.join(data_path, filename)
       

        
        if recfile.endswith('.bin'):

            #get cell number 
            cell_num = recfile.split('_')[3].split('/')[1]
            file_type = recfile.split('_')[4]
           
            
   
            
            #if neuropixel file, figure out if lfp or spike file
            if file_type == "npx":
                file_type = recfile.split('_')[5].split('.')[0]
            
          
            if cell_num in cell_ids:
               
                #if lfp file 

                if file_type == 'lfp':
                  
                    #Load LFP data
                    lfp_recording = np.memmap(recfile, mode = 'r', dtype=np.int16, order = 'C')
                    lfp_samples = int(len(lfp_recording)/npx_channels)
                    lfp_recording = lfp_recording.reshape((lfp_samples, npx_channels), order = 'F')
                    lfp_recording = lfp_recording[:, npx_patch_channel]
                    lfp_recording = np.asarray(lfp_recording)
                    lfp_recording = lfp_recording - np.mean(lfp_recording) # de-mean
                    # # filter the LFP data
                    lfp_filt = butter_bandpass(lfp_recording, fs=lfp_fs, filt_freq=[0.1, 250], order=4)

                    lfp_times = np.arange(0, np.shape(lfp_recording)[0]) / lfp_one_ms # in ms instead of sec
                    
                 
                    
                    np.save(out_lfp_path + cell_num + '_lfp_filt.npy', lfp_filt)

                #if patch file
                
                
                if file_type == 'patch':
                    
                    
                    #get patch fs of that cell
                    patch_one_ms = float(patch_fs/1000)
                    #get patch data 
                    patch_recording = np.fromfile(recfile, dtype='float64')
                    patch_recording = np.asarray(patch_recording)
                    #patch_recording = -patch_recording # positive up


                    patch_times = np.arange(0, np.shape(patch_recording)[0]) / patch_one_ms # in ms instead of sec
                    

                    # filter the patch data
                    patch_filt = butter_bandpass(patch_recording, fs=patch_fs, filt_freq=[10, 25000], order=4)
                    
                    np.save(out_patch_path + cell_num + 'patch_filt.npy', patch_filt)
                
                #if npx spiking file 
                
                
                if file_type == 'raw':
                    
                    # load npx data
                    npx_recording = np.memmap(recfile, mode = 'r', dtype=np.int16, order = 'C')
          
                    npx_samples = int(len(npx_recording)/npx_channels)
                    npx_recording = npx_recording.reshape((npx_channels, npx_samples), order = 'F')
                    npx_recording = npx_recording[npx_patch_channel, :]
                    npx_recording = np.asarray(npx_recording)
                    
                    
                    npx_times = np.arange(0, np.shape(npx_recording)[0]) / npx_one_ms # in ms instead of sec
                    
                    # filter the neuropixels data
                    npx_filt = butter_bandpass(npx_recording, fs=npx_fs, filt_freq=[300, 14000], order=4)
                    np.save(out_npx_path + cell_num + 'npx_filt.npy', npx_filt)
                    
                
               

    return lfp_recording, lfp_times, patch_filt, patch_times



In [5]:
def get_patch_fs_metadata(metadata_file, cell_num):
    
    metafile = pd.read_excel(metadata_file, sheet_name='Sheet1')
    #get index for cell
    cells = metafile['Cell']
    index_cell = cells[cells == "c"+str(cell_num)].index[0]
    
    #get path fs for cell
    
    patch_fs = metafile['Patch Samp. Freq. (Hz)'][index_cell]
    
    return patch_fs
    

### Create dictionaries for fs for cell and npx channel closest to soma for each cell

In [6]:
cell_num_list = [6, 7,8,14, 15, 16, 18, 19, 20, 21, 22, 24, 27, 42,44,45]
patch_fs_list = [50023.89817284,50023.91361167,50023.90949465,50023.89868747,50023.89405582,50023.90270156,50023.89199731,
                 50023.88376327,50023.87964625,50023.89714358,50023.89474199,50023.89405582,50023.88788029,50023.88170476,50023.87244147,50023.87347073]

chan_pred_list = [172,188,188,159,183,220,157,178,199,180,205,248,223,108,178,178]



In [7]:
dict_patch_fs = dict(zip(cell_num_list, patch_fs_list))

dict_chan_pred = dict(zip(cell_num_list, chan_pred_list ))

In [8]:
dict_chan_pred

{6: 172,
 7: 188,
 8: 188,
 14: 159,
 15: 183,
 16: 220,
 18: 157,
 19: 178,
 20: 199,
 21: 180,
 22: 205,
 24: 248,
 27: 223,
 42: 108,
 44: 178,
 45: 178}

In [12]:
cell_ids = 'c14'
cell_num =14

npx_fs = 30000 # sampling rate
lfp_fs = 2500 # sampling rate

patch_fs = dict_patch_fs[cell_num ]

npx_channels = 384
npx_patch_channel = dict_chan_pred[cell_num ]

data_path = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/Neuropixel Paired Recordings/Recordings/all_recordings'

out_lfp_path = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/Neuropixel Paired Recordings/Recordings/filt_lfp_recordings/'


out_npx_path = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/Neuropixel Paired Recordings/Recordings/filt_npx_recordings/'

out_patch_path = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/Neuropixel Paired Recordings/Recordings/filt_patch_recordings/'




In [13]:
lfp_recording, lfp_times, patch_filt, patch_times = load_spe1_data(data_path, lfp_fs, npx_fs, patch_fs, cell_ids, npx_channels, npx_patch_channel, out_lfp_path, out_npx_path, out_patch_path)


  0%|          | 0/27 [00:00<?, ?it/s]